## Notebook 2 — `02_eda.ipynb`

**Cell 1 — Markdown**
# Exploratory Data Analysis
Six analysis areas covering sales trends, product performance, regional breakdown,
category profitability, discount effects, and seasonality.

In [2]:
import sys
sys.path.append("..")

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from src.data_loader import load_data, clean_data

df = clean_data(load_data("../data/superstore.csv"))

PALETTE = ["#1F4E79", "#2E75B6", "#4BACC6", "#A9CCE3", "#D6E4F0"]
TEMPLATE = "plotly_white"

Shape: (9994, 21)

Column dtypes:
row_id                    int64
order_id                 object
order_date       datetime64[ns]
ship_date        datetime64[ns]
ship_mode                object
customer_id              object
customer_name            object
segment                  object
country                  object
city                     object
state                    object
postal_code               int64
region                   object
product_id               object
category                 object
sub_category             object
product_name             object
sales                   float64
quantity                  int64
discount                float64
profit                  float64
dtype: object

Null counts:
row_id           0
order_id         0
order_date       0
ship_date        0
ship_mode        0
customer_id      0
customer_name    0
segment          0
country          0
city             0
state            0
postal_code      0
region           0
product_id       0



## 1. Monthly Sales Trend
Question: Is revenue growing year over year? Are there seasonal spikes?

In [3]:
monthly = (
    df.groupby(["year", "month"])["sales"]
    .sum()
    .reset_index()
)
monthly["date"] = pd.to_datetime(monthly[["year", "month"]].assign(day=1))
monthly = monthly.sort_values("date")

fig = px.line(
    monthly,
    x="date",
    y="sales",
    title="Monthly Sales Revenue (2014–2017)",
    labels={"date": "Month", "sales": "Total Sales ($)"},
    template=TEMPLATE,
    color_discrete_sequence=PALETTE,
    markers=True
)
fig.update_layout(hovermode="x unified")
fig.show()

## 2. Top 10 Products by Revenue
Question: Which specific products drive the most revenue?

In [4]:
top_products = (
    df.groupby("product_name")["sales"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

fig = px.bar(
    top_products,
    x="sales",
    y="product_name",
    orientation="h",
    title="Top 10 Products by Revenue",
    labels={"sales": "Total Sales ($)", "product_name": "Product"},
    template=TEMPLATE,
    color="sales",
    color_continuous_scale="Blues"
)
fig.update_layout(yaxis=dict(autorange="reversed"), coloraxis_showscale=False)
fig.show()

## 3. Sales & Profit by Region
Question: Which region leads in revenue vs profit? Is there a mismatch?

In [5]:
region = (
    df.groupby("region")[["sales", "profit"]]
    .sum()
    .reset_index()
    .melt(id_vars="region", var_name="metric", value_name="value")
)

fig = px.bar(
    region,
    x="region",
    y="value",
    color="metric",
    barmode="group",
    title="Sales vs Profit by Region",
    labels={"value": "Amount ($)", "region": "Region", "metric": "Metric"},
    template=TEMPLATE,
    color_discrete_sequence=["#1F4E79", "#4BACC6"]
)
fig.show()

## 4. Category & Sub-Category Profitability
Question: Where is profit concentrated vs where is revenue concentrated?


In [6]:
subcat = (
    df.groupby(["category", "sub_category"])[["sales", "profit"]]
    .sum()
    .reset_index()
)

fig = px.treemap(
    subcat,
    path=["category", "sub_category"],
    values="sales",
    color="profit",
    color_continuous_scale=["#d73027", "#fee08b", "#1a9850"],
    color_continuous_midpoint=0,
    title="Revenue by Sub-Category (colour = profit, red = loss)",
    template=TEMPLATE
)
fig.update_traces(textinfo="label+value")
fig.show()


## 5. Discount Effect on Profit Margin
Question: Does higher discount reliably destroy profit margin?

In [7]:
fig = px.scatter(
    df,
    x="sales",
    y="profit",
    color="discount",
    color_continuous_scale="RdBu_r",
    title="Sales vs Profit — Coloured by Discount Level",
    labels={"sales": "Sales ($)", "profit": "Profit ($)", "discount": "Discount"},
    template=TEMPLATE,
    opacity=0.6,
    hover_data=["product_name", "sub_category", "region"]
)
fig.add_hline(y=0, line_dash="dash", line_color="red", annotation_text="Break-even")
fig.show()


## 6. Monthly Sales Seasonality by Category
Question: Do all categories follow the same seasonal pattern or do they diverge?

In [8]:
cat_monthly = (
    df.groupby(["year", "month", "category"])["sales"]
    .sum()
    .reset_index()
)
cat_monthly["date"] = pd.to_datetime(cat_monthly[["year", "month"]].assign(day=1))
cat_monthly = cat_monthly.sort_values("date")

fig = px.line(
    cat_monthly,
    x="date",
    y="sales",
    color="category",
    title="Monthly Sales by Category (2014–2017)",
    labels={"date": "Month", "sales": "Total Sales ($)", "category": "Category"},
    template=TEMPLATE,
    color_discrete_sequence=PALETTE,
    markers=True
)
fig.update_layout(hovermode="x unified")
fig.show()

## Key Business Insights

1. **Revenue is growing but heavily seasonal.** Monthly sales show a clear upward trend from
   2014 to 2017, with the peak month reaching ~$118k in late 2017. However, every year shows
   a sharp Q4 spike followed by a steep January drop. — Implication: the business is
   over-reliant on Q4 demand. — Recommendation: introduce mid-year promotions in Q2 to
   smooth revenue distribution.

2. **A single product dominates the top 10.** The Canon imageCLASS 2200 Copier generates
   ~$60k in revenue, nearly double the second-ranked product. The remaining 9 products are
   tightly clustered between $15k–$30k. — Implication: revenue concentration in one SKU is
   a supply chain risk. — Recommendation: monitor stock levels for the Canon copier closely
   and identify what drives its outsized demand.

3. **West leads in both sales and profit, Central is the weakest performer.** West generates
   the highest revenue (~$710k) and the highest profit (~$105k). Central has ~$500k in sales
   but only ~$30k in profit — the worst profit-to-sales ratio of all regions. — Implication:
   Central is likely over-discounting or carrying low-margin products. — Recommendation:
   audit discount rates and product mix in the Central region specifically.

4. **Tables is the only sub-category operating at a loss.** The treemap shows Tables as the
   only clearly loss-making sub-category (orange/red) despite significant revenue (~$207k).
   Technology sub-categories (Phones, Accessories, Copiers) are the most profitable.
   — Implication: every Tables sale is actively destroying profit. — Recommendation: urgently
   review the discount and pricing strategy for Tables, or consider deprioritising it.

5. **High discounts are directly causing losses.** The scatter plot shows a clear pattern:
   red dots (discount > 0.4) almost exclusively sit below the break-even line, while blue
   dots (low discount) cluster in positive profit territory. — Implication: the discount
   strategy is not driving enough volume to offset margin loss. — Recommendation: cap
   discounts at 20% and A/B test the impact on order volume vs profit.

6. **All three categories spike in Q4, but Technology spikes the hardest.** November is the
   peak month for every category, but Technology shows the sharpest single-month jump
   (~$50k in Nov 2017). Office Supplies is the most stable and predictable year-round.
   — Implication: Technology demand is highly event-driven (Black Friday, year-end budgets).
   — Recommendation: ensure Technology inventory is stocked by October and plan targeted
   campaigns around November for that category.